# Final Pipeline for Text Data

Goal is to:
1. Combine all preprocessing steps into one pipeline
2. Apply it to the IMDb dataset
3. Save cleaned dataset for modeling

In [1]:
# Step 1: Import libraries

import os
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK resources (first run only)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /Users/mimi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/mimi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/mimi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/mimi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/mimi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
# Step 2: Define preprocessing pipeline

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, use_stemming=False):
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    
    # Tokenization
    # tokens = word_tokenize(text)
    tokens = text.split()  # Simple fallback tokenizer

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Stemming or Lemmatization
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    else:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return " ".join(tokens)  # join back into text

In [3]:
# Step 3: Load dataset (subset for demo, can increase later)

base_dir = "/Users/mimi/Desktop/nlp project/aclImdb"
train_pos_dir = os.path.join(base_dir, "train/pos")
train_neg_dir = os.path.join(base_dir, "train/neg")

def load_reviews(directory, label, limit=1000):
    data = []
    for i, fname in enumerate(os.listdir(directory)):
        if i >= limit:
            break
        with open(os.path.join(directory, fname), encoding="utf-8") as f:
            data.append((f.read(), label))
    return data

pos_reviews = load_reviews(train_pos_dir, 1, limit=1000)
neg_reviews = load_reviews(train_neg_dir, 0, limit=1000)

all_data = pos_reviews + neg_reviews
df = pd.DataFrame(all_data, columns=["review", "label"])

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2000, 2)


,review,label
0,For a movie that gets no respect there sure ar...,1
1,Bizarre horror movie filled with famous faces ...,1
2,"A solid, if unremarkable film. Matthau, as Ein...",1
3,It's a strange feeling to sit alone in a theat...,1
4,"You probably all already know this by now, but...",1


In [4]:
# Step 4: Apply preprocessing

df["cleaned_review"] = df["review"].apply(lambda x: preprocess_text(x, use_stemming=False))

df.head(10)


,review,label,cleaned_review
0,For a movie that gets no respect there sure ar...,1,movie get respect sure lot memorable quote lis...
1,Bizarre horror movie filled with famous faces ...,1,bizarre horror movie filled famous face stolen...
2,"A solid, if unremarkable film. Matthau, as Ein...",1,solid unremarkable film matthau einstein wonde...
3,It's a strange feeling to sit alone in a theat...,1,strange feeling sit alone theater occupied par...
4,"You probably all already know this by now, but...",1,probably already know 5 additional episode nev...
5,I saw the movie with two grown children. Altho...,1,saw movie two grown child although clever shre...
6,You're using the IMDb.<br /><br />You've given...,1,youre using imdbbr br youve given hefty vote f...
7,This was a good film with a powerful message o...,1,good film powerful message love redemption lov...
8,"Made after QUARTET was, TRIO continued the qua...",1,made quartet trio continued quality earlier fi...
9,"For a mature man, to admit that he shed a tear...",1,mature man admit shed tear film mature respons...


In [5]:
# Step 5: Save cleaned dataset

df.to_csv("cleaned_imdb_reviews.csv", index=False)
print(" Cleaned dataset saved as cleaned_imdb_reviews.csv")


 Cleaned dataset saved as cleaned_imdb_reviews.csv


# After running this, you should have a CSV file with:
- ``review`` (original text)
- ``label`` (0=negative, 1=positive)
- ``cleaned_review`` (fully preprocessed text)